# UC Atlas: Compositional Analysis with scCODA

Bayesian compositional analysis of cell type abundance across UC conditions
(healthy, non-inflamed, inflamed) using scCODA via pertpy.

Analysis restricted to datasets with matched paired samples (P1, P4, P5, P8)
to reduce confounding by patient-level variation.
Reference cell type: cycling epithelial (least condition-varying).
Results visualized as a forest plot of credible intervals (Fig. 1E / Ext. Fig.).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pertpy as pt
import scvi

scvi.settings.dl_num_workers = 0

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
OUTPUT_DIR = f"{DATA_DIR}/sccoda"

## 1. Load and filter atlas

In [ ]:
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")

# Restrict to datasets with complete multi-condition sampling
scrna = scrna[scrna.obs["orig.ident"].isin(["P1", "P4", "P5", "P8"])]
print(scrna.obs["cell_type_short"].value_counts())

## 2. Prepare scCODA model

In [ ]:
scc = pt.tl.Sccoda()

# Aggregate to sample level and build MuData
mdata = scc.load(
    scrna,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell_type_short",
    sample_identifier="sample",
    covariate_obs=["condition"]
)

# Prepare with condition as the covariate; cycling epithelial as reference
mdata = scc.prepare(
    mdata,
    formula="C(condition)",
    reference_cell_type="Cycl Epi"
)

## 3. Run NUTS MCMC sampling

In [ ]:
# NUTS sampling (11 000 steps; first 1 000 warm-up)
scc.run_nuts(mdata)

## 4. Extract and save results

In [ ]:
scc.summary(mdata, extended=True)

eff  = scc.get_effect_df(mdata)   # posterior coefficients per cell type × covariate
cred = scc.credible_effects(mdata)  # binary: credibly non-zero

eff.to_csv(f"{OUTPUT_DIR}/sccoda_effect_df.csv")
print(eff)

## 5. Forest plot: credible interval per condition (Fig. 1E)

In [ ]:
df = eff.reset_index() if isinstance(eff.index, pd.MultiIndex) else eff

# Rename index columns if needed
rename_map = {c: c.title().replace(" ", " ") for c in df.columns}
for col in df.columns:
    if col.lower() == "covariate":   rename_map[col] = "Covariate"
    if col.lower().replace(" ", "_") in ("cell_type", "celltype"): rename_map[col] = "Cell Type"
df = df.rename(columns=rename_map)

cov_inflamed    = "C(condition)T.Inflamed"
cov_noninflamed = "C(condition)T.Non-inflamed"
df = df[df["Covariate"].isin([cov_inflamed, cov_noninflamed])].copy()

for c in ("Final Parameter", "HDI 3%", "HDI 97%"):
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Order cell types by inflamed HDI upper bound
order = (
    df[df["Covariate"] == cov_inflamed]
    .sort_values("HDI 97%", ascending=True)["Cell Type"]
    .tolist()
)
order += sorted(ct for ct in df["Cell Type"].unique() if ct not in order)

piv = df.pivot(index="Cell Type", columns="Covariate",
               values=["Final Parameter", "HDI 3%", "HDI 97%"]).reindex(order)

y = np.arange(len(piv.index))
offset = 0.15

def arr(metric, cov):
    return piv[(metric, cov)].to_numpy()

def xerr(est, lo, hi):
    return np.vstack([est - lo, hi - est])

fig, ax = plt.subplots(figsize=(6, 0.27 * len(piv.index)))

ax.errorbar(
    arr("Final Parameter", cov_inflamed), y - offset,
    xerr=xerr(arr("Final Parameter", cov_inflamed),
              arr("HDI 3%", cov_inflamed), arr("HDI 97%", cov_inflamed)),
    fmt="o", capsize=2, elinewidth=1, markersize=4,
    color="#ab210c", ecolor="#ab210c", alpha=0.9, label="Inflamed"
)
ax.errorbar(
    arr("Final Parameter", cov_noninflamed), y + offset,
    xerr=xerr(arr("Final Parameter", cov_noninflamed),
              arr("HDI 3%", cov_noninflamed), arr("HDI 97%", cov_noninflamed)),
    fmt="o", capsize=2, elinewidth=1, markersize=4,
    color="#a88c0f", ecolor="#a88c0f", alpha=0.9, label="Non-inflamed"
)

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(piv.index)
ax.set_ylim(-0.5 - offset, (len(piv.index) - 0.5) + offset)
ax.invert_yaxis()
ax.margins(y=0)
ax.set_xlabel("Final parameter (3–97% HDI)")
ax.set_title("UC scCODA: Condition Effects by Cell Type")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sccoda_forest_plot.pdf", bbox_inches="tight", dpi=300)
plt.show()